# Math RL: GSM8K Word Problems

This notebook demonstrates training an LLM to solve grade-school math word problems using reinforcement learning.

**Goal**: Train the model to solve multi-step word problems from the GSM8K dataset.

**Difficulty**: Much harder than arithmetic - requires reasoning, not just computation.

In [ ]:
import os
os.environ['TINKER_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

## Dataset: GSM8K

GSM8K (Grade School Math 8K) contains:
- 8.5K training problems
- Multi-step word problems (2-8 steps)
- Natural language requiring extraction of quantities

### Example Problem
```
Question: Janet's ducks lay 16 eggs per day. She eats three for breakfast 
every morning and bakes muffins for her friends every day with four. She 
sells the remainder at the farmers' market daily for $2 per fresh duck egg. 
How much in dollars does she make every day at the farmers' market?

Answer: 18
```

In [ ]:
# Training configuration
config = {
    'model_name': 'meta-llama/Llama-3.2-1B',
    'env': 'gsm8k',
    'group_size': 4,           # Samples per problem
    'groups_per_batch': 50,    # Problems per batch
    'learning_rate': 1e-4,
    'lora_rank': 32,
    'max_tokens': 512,         # Longer for reasoning
}

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")

## Run Training

In [ ]:
!python -m tinker_cookbook.recipes.math_rl.train \
    model_name="meta-llama/Llama-3.2-1B" \
    env=gsm8k \
    group_size=4 \
    groups_per_batch=50 \
    learning_rate=1e-4 \
    max_tokens=512

## GSM8K vs Arithmetic

| Aspect | Arithmetic | GSM8K |
|--------|-----------|-------|
| Complexity | Single operation | Multi-step reasoning |
| Input | "What is 45 + 32?" | Long word problem |
| Tokens needed | 5 | 512+ |
| Learning speed | ~20 steps to 100% | Much slower |
| Baseline accuracy | ~70% | ~10-20% |

## Understanding the Output

### Trajectory Examples
```
Question: A farmer has 120 apples...

trajectory idx=0, reward=1
  correct: 1.0 (right answer)
  
trajectory idx=1, reward=0
  correct: 0.0 (wrong answer)
```

### Key Metrics
| Metric | Description |
|--------|-------------|
| `env/all/correct` | Accuracy on word problems |
| `env/all/reward/total` | Average reward |
| `env/all/format` | Format compliance |
| `ac_tokens_per_turn` | Tokens used for reasoning |

## Expected Learning Curve

GSM8K is challenging - expect slower improvement:

```
Step   0: correct=15%
Step  50: correct=25%
Step 100: correct=35%
Step 200: correct=45%
```

Note: Larger models (8B+) achieve much higher accuracy.

## Available Math Environments

The math_rl recipe supports multiple environments:

In [ ]:
environments = {
    'arithmetic': 'Simple addition (trivial)',
    'gsm8k': 'Grade school word problems',
    'math': 'Competition math (MATH dataset)',
}

print("Available environments:")
for env, desc in environments.items():
    print(f"  {env}: {desc}")

## Analyze Results

In [ ]:
import json
import glob

# Find the latest metrics file
metrics_files = glob.glob('/tmp/tinker-examples/math_rl/gsm8k-*/metrics.jsonl')
if metrics_files:
    latest = max(metrics_files)
    print(f"Reading: {latest}\n")
    
    accuracy_history = []
    
    with open(latest) as f:
        for line in f:
            data = json.loads(line)
            if 'env/all/correct' in data:
                accuracy_history.append(data['env/all/correct'] * 100)
    
    if accuracy_history:
        print(f"Starting accuracy: {accuracy_history[0]:.1f}%")
        print(f"Current accuracy: {accuracy_history[-1]:.1f}%")
        print(f"Improvement: +{accuracy_history[-1] - accuracy_history[0]:.1f}%")